## Setup

In [1]:
# autoreload so we don't have to restart the kernel when we change code in rag_helper.py
%load_ext autoreload
%autoreload 2

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

## Q1. How many lesson pages
How many lesson pages are in the dataset?

- 24
- 72 ✅
- 240
- 720

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']  
)

index.fit(documents)


In [6]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
print(index.search(question)[0]['content'][:30])
print(index.search(question)[0]['filename'])

# The Agentic Loop

Video: [Wa
01-agentic-rag/lessons/14-agentic-loop.md


## Q2. Indexing and searching

Index the documents with minsearch - make content a text field and
filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/14-agentic-loop.md ✅
- 04-evaluation/lessons/13-llm-as-judge.md
- 06-best-practices/lessons/02-hybrid-search.md

In [8]:
from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [9]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

# extra funcionality to get the last usage of the model
response = assistant.rag(question, return_usage=True)

In [10]:
# dedicated method to get the last usage of the model
assistant.get_last_usage()

ResponseUsage(input_tokens=623, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=716)

In [11]:
assistant.get_last_token_counts()

{'input_tokens': 623, 'output_tokens': 93, 'total_tokens': 716}

## Q3. How many input (prompt) tokens did we send to the model for
this request?

* 700 ✅ (approx 623)
* 7000
* 70000
* 700000

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

## Q4: How many chunks do you get?

- 70
- 295 ✅
- 1100
- 4500

In [15]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

chunk_index.fit(chunks)

In [16]:
chunk_index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [17]:
print(chunk_index.search(question)[0]['content'][:30])
print(chunk_index.search(question)[0]['filename'])

while` loop. The loop keeps ca
01-agentic-rag/lessons/14-agentic-loop.md


In [18]:
assistant_chunck = RAGBase(
    index=chunk_index,
    llm_client=openai_client
)

# extra funcionality to get the last usage of the model
response = assistant_chunck.rag(question, return_usage=True)

In [24]:
print(assistant_chunck.get_last_usage())
print("\n")
print(assistant_chunck.get_last_token_counts())

ResponseUsage(input_tokens=477, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=73, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=550)


{'input_tokens': 477, 'output_tokens': 73, 'total_tokens': 550}


In [28]:
assistant.get_last_usage().input_tokens / assistant_chunck.get_last_usage().input_tokens

1.3060796645702306

## Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

- about the same (+/-30% less from 623 -> 477) ✅
- 3× fewer
- 10× fewer
- 30× fewer